In [16]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [17]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [18]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [19]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your operating system:
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and start a model locally:
   ```bash
   ollama run llama3
   ```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

3. To test that the local server is running, use:
   ```bash
   curl http://localhost:11434
   ```

If you want to use it from Python, install the client with:
```bash
pip install ollama
```


# Function Calling

In [20]:
answer = assistant.rag('How do I run Oyama locally?')
print(answer)

You can run Oyama locally if you’re comfortable setting up the needed tools yourself. The course FAQ says local setup is fine instead of Codespaces, but you’ll need to set up Python, `uv`, Jupyter, Docker, and any other tools required for the module.

If you do run it locally, make sure to:
- document your setup
- keep your environment reproducible




In [21]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Possibly — but it depends on the course’s enrollment rules and whether registration is still open.\n\nIf you want, I can help you figure it out. Usually you’d check:\n- the course start date\n- whether late registration is allowed\n- prerequisite requirements\n- whether there’s still capacity\n- if the instructor or registrar can approve a late add\n\nIf you mean a specific course, send me the course name or a link and I can help you draft a message asking to join.'

In [22]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [23]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [24]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [25]:
len(response.output)

1

In [26]:
call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join course registration after start"}', call_id='call_dMUiK7hZHvR1bath1DzqXfGx', name='search', type='function_call', id='fc_09d3721aa53b7e88006a3600c7592881979b3e9822dc9a91ec', namespace=None, status='completed')

In [27]:
import json

args = json.loads(call.arguments)
args

{'query': 'join course discovered late enrollment can I join course registration after start'}

In [28]:
call.name

'search'

In [29]:
results = search(**args)

In [30]:
result_json = json.dumps(results, indent=2)

In [31]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [32]:
messages.append(call)
messages.append(function_call_output)
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join course registration after start"}', call_id='call_dMUiK7hZHvR1bath1DzqXfGx', name='search', type='function_call', id='fc_09d3721aa53b7e88006a3600c7592881979b3e9822dc9a91ec', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_dMUiK7hZHvR1bath1DzqXfGx',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workfl

In [33]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)
print(response.output_text)

Yes — you can still join.

You don’t need special approval or a confirmation email. You can start learning and submitting homework while the forms are open.

If you want a certificate, make sure you submit your project while the course is still accepting submissions.


In [34]:
usage = response.usage
usage.input_tokens, usage.output_tokens
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


# Agentic Loop

In [35]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [36]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [37]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course enrollment discovered course can I join"}
function_call: search {"query":"course registration late enrollment discovered course can I join"}


In [38]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join.

You can start learning and doing the materials even if you discovered the course late. If you want a certificate, though, you need to submit your project while submissions are still open, and certificates are only available with a live cohort, not self-paced.

If you want, I can also help you figure out the best way to start catching up. Any other areas you want to explore?


In [39]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [40]:
agent_loop(instructions, "How do I run Olam locally?")

iteration #1...
function_call: search {"query":"run Olam locally"}
iteration #2...
function_call: search {"query":"Olam locally Ollama local run install ollama server localhost:11434 Python client"}
iteration #3...
ASSISTANT:
If you mean **Ollama**: yes, you can run it locally.

### Install Ollama
- **macOS / Windows**: download and install from https://ollama.com/download
- **Linux**:
```bash
curl -fsSL https://ollama.com/install.sh | sh
```

### Start a model locally
```bash
ollama run llama3
```
This will download the model and start a local chat session.

### Check that the local server is running
```bash
curl http://localhost:11434
```
You should get a response showing the available models.

### Use it from Python
```bash
pip install ollama
```

```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": "Hello!"}]
)

print(response['message']['content'])
```

### If you get a connection refused error
Restart the server with:
```b

'If you mean **Ollama**: yes, you can run it locally.\n\n### Install Ollama\n- **macOS / Windows**: download and install from https://ollama.com/download\n- **Linux**:\n```bash\ncurl -fsSL https://ollama.com/install.sh | sh\n```\n\n### Start a model locally\n```bash\nollama run llama3\n```\nThis will download the model and start a local chat session.\n\n### Check that the local server is running\n```bash\ncurl http://localhost:11434\n```\nYou should get a response showing the available models.\n\n### Use it from Python\n```bash\npip install ollama\n```\n\n```python\nimport ollama\n\nresponse = ollama.chat(\n    model=\'llama3\',\n    messages=[{"role": "user", "content": "Hello!"}]\n)\n\nprint(response[\'message\'][\'content\'])\n```\n\n### If you get a connection refused error\nRestart the server with:\n```bash\nollama serve\n```\nIf needed in a notebook:\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```\n\nIf you meant something else by “Olam,” let me know and I’ll adjust. Would 

In [41]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late join signup"}
iteration #2...
function_call: search {"query":"certificate submit project while accepting submissions live cohort peer review deadlines self-paced join course now"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while the course is still accepting submissions. Also, certificates are only available for the live cohort, not self-paced study.

If you’d like, I can also help you understand how registration, homework, and certificates work. Is there anything else you want to explore?


'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while the course is still accepting submissions. Also, certificates are only available for the live cohort, not self-paced study.\n\nIf you’d like, I can also help you understand how registration, homework, and certificates work. Is there anything else you want to explore?'

In [42]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess definition opening"}
iteration #2...
function_call: search {"query":"queen gambit chess opening move d4 d5 c4 FAQ"}
iteration #3...
ASSISTANT:
A **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** as a gambit to try to gain control of the center, especially the **e4** square. It’s one of the oldest and most famous chess openings.

There are two main versions:
- **Queen’s Gambit Accepted**: Black takes the pawn on c4
- **Queen’s Gambit Declined**: Black does not take it and instead keeps the pawn structure

If you want, I can also explain:
- why it’s called a “gambit”
- the basic ideas for White and Black
- the difference between accepted and declined lines


'A **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** as a gambit to try to gain control of the center, especially the **e4** square. It’s one of the oldest and most famous chess openings.\n\nThere are two main versions:\n- **Queen’s Gambit Accepted**: Black takes the pawn on c4\n- **Queen’s Gambit Declined**: Black does not take it and instead keeps the pawn structure\n\nIf you want, I can also explain:\n- why it’s called a “gambit”\n- the basic ideas for White and Black\n- the difference between accepted and declined lines'

In [43]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen gambit” or “queen’s gambit chess opening,” so this looks off-topic for the course FAQ.

If you meant something specific to the course, feel free to rephrase it with course-related terms. Are there other areas you want to explore?


'I couldn’t find any course FAQ entry about “queen gambit” or “queen’s gambit chess opening,” so this looks off-topic for the course FAQ.\n\nIf you meant something specific to the course, feel free to rephrase it with course-related terms. Are there other areas you want to explore?'

# ToyAIKit

In [44]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [45]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [46]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [47]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [48]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [49]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [50]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [51]:
result.cost

CostInfo(input_cost=Decimal('0.00300525'), output_cost=Decimal('0.0014265'), total_cost=Decimal('0.00443175'))

In [52]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama run locally"}', call_id='call_FXlr2BFMffTgQyBERjHpGpPG',

In [53]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [ ]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received
